# Stage 1: Data Preprocessing & Feature Extraction
In this notebook, we handle the first stages of the Spatio-Temporal SAR Architecture:
1. **Data Input**: Loading SAR Image Sequences from the Sen1Floods11 dataset.
2. **Preprocessing**: Radiometric calibration (clipping), Normalization, and **Speckle Noise Reduction**.
3. **Feature Extraction**: Training a Convolutional Autoencoder (CAE) to capture important spatial features.
The trained Encoder will be saved for Stage 2 (Temporal Modeling).

In [1]:
import os
import torch
from torch.utils.data import DataLoader
from dataset import Sen1FloodsDataset, get_train_val_test_splits
from models import ConvAutoencoder
from train import train_autoencoder
from utils import set_seed, load_config

# 1. Setup & Configuration
config = load_config('config.json')
set_seed(config['training']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


In [2]:
# 2. Data Loading
import rasterio

s1_dir = config['data']['s1_dir']
label_dir = config['data']['label_dir']

# Filter only valid TIF files
if os.path.exists(s1_dir) and os.path.exists(label_dir):
    s1_files = sorted([f for f in os.listdir(s1_dir) if f.endswith('.tif')])
    label_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.tif')])
    file_pairs = list(zip([os.path.join(s1_dir, f) for f in s1_files], 
                          [os.path.join(label_dir, f) for f in label_files]))
else:
    file_pairs = []
    print("Warning: Data directories not found.")

if len(file_pairs) > 0:
    # 70/15/15 Split
    train_idx, val_idx, test_idx = get_train_val_test_splits(len(file_pairs), config['data']['split_ratio'])
    
    train_pairs = [file_pairs[i] for i in train_idx]
    val_pairs = [file_pairs[i] for i in val_idx]
    
    train_dataset = Sen1FloodsDataset(train_pairs, target_size=tuple(config['data']['target_size']))
    val_dataset = Sen1FloodsDataset(val_pairs, target_size=tuple(config['data']['target_size']))
    
    train_loader = DataLoader(train_dataset, batch_size=config['training']['batch_size'], shuffle=True, num_workers=config['data']['num_workers'])
    val_loader = DataLoader(val_dataset, batch_size=config['training']['batch_size'], shuffle=False, num_workers=config['data']['num_workers'])
    
    print(f"Loaded {len(train_dataset)} training and {len(val_dataset)} validation samples.")
else:
    print("No data pairs available. Please run data download.")


Loaded 28 training and 6 validation samples.


In [3]:
# 3. Model Definition & Training
if len(file_pairs) > 0:
    model = ConvAutoencoder(dropout_rate=config['model']['dropout_rate']).to(device)
    
    # Execute full training loop with validation and early stopping
    train_autoencoder(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        num_epochs=config['training']['autoencoder_epochs'], 
        lr=config['training']['autoencoder_lr'],
        device=device
    )


Starting Autoencoder Training...


Epoch [1/15], Train Loss: 0.03837, Val Loss: 0.03915


Epoch [2/15], Train Loss: 0.02933, Val Loss: 0.03846


Epoch [3/15], Train Loss: 0.02410, Val Loss: 0.03757


Epoch [4/15], Train Loss: 0.02209, Val Loss: 0.03625


Epoch [5/15], Train Loss: 0.01948, Val Loss: 0.03471


... (6 outputs left out)

Epoch [12/15], Train Loss: 0.01040, Val Loss: 0.02514


Epoch [13/15], Train Loss: 0.00911, Val Loss: 0.02516


Epoch [14/15], Train Loss: 0.00782, Val Loss: 0.02478


Epoch [15/15], Train Loss: 0.00731, Val Loss: 0.02335
Autoencoder Training Complete. Best model saved to models/best_autoencoder.pth
